In [ ]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

In [65]:
def fetch_banapresso():
    options = Options()
    options.add_argument("--start-maximized")
    url = "https://www.banapresso.com/"
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(1.5)

    action = ActionChains(driver)

    # 매장 > 매장 찾기
    first_tag = driver.find_element(By.CSS_SELECTOR, "#wrap > header > div > ul > li:nth-child(2) > a")
    second_tag = driver.find_element(By.CSS_SELECTOR, "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a")
    action.move_to_element(first_tag).move_to_element(second_tag).click().perform()
    print("매장 찾기 선택")

    # 위치 정보 권한 허용 팝업 확인 버튼 클릭
    try:
        popup_tag = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((
                By.CSS_SELECTOR, "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm > button"
            ))
        )
        popup_tag.click()
        print("팝업 닫기 완료")
        time.sleep(1)
    except:
        print("위치 정보 권한 허용 팝업이 없거나 이미 닫혀 있습니다.")

    # 끝까지 스크롤
    while True:
        before = driver.execute_script("return document.querySelector('#store div.store_shop_list').scrollTop")
        driver.execute_script("document.querySelector('#store div.store_shop_list').scrollTop += 10000")
        time.sleep(2)
        after = driver.execute_script("return document.querySelector('#store div.store_shop_list').scrollTop")
        if before == after:
            break

    # html로 파서 후 li 태그 검색
    req = driver.page_source
    soup = BeautifulSoup(req, "html.parser")
    stores = soup.select_one("#store div.store_shop_list > ul").find_all('li')
    
    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    store_time_list = []

    # 데이터 추출
    for store in stores:
        try:
            store_name = store.select_one("p.name").text
            store_addr = store.select_one("p.address").text
            store_time = store.find("div", class_="time").find_all("p")[1].text

            # 각 데이터 저장 리스트에 추가
            store_list.append(store_name)
            addr_list.append(store_addr)
            store_time_list.append(store_time)
        except:
            continue

    df = pd.DataFrame({
        '매장명': store_list,
        '주소': addr_list,
        '영업 시간': store_time_list
    })
    df.index = df.index + 1

    driver.quit()

    return df


In [66]:
# banapresso_df = fetch_banapresso()

# banapresso_df.to_csv(
#     "0625work_banapresso_store_list.csv",
#     encoding='utf-8-sig'
# )

# print("데이터가 0625work_banapresso_store_list.csv 파일로 저장되었습니다.")

df = fetch_banapresso()
print(df)
print(len(df))

매장 찾기 선택
팝업 닫기 완료
            매장명                              주소        영업 시간
1          가락몰점   서울특별시 송파구 양재대로 932, 업무동 1층 로비  07:30~23:30
2     가산디지털단지역점               서울시 금천구 가산동 60-3   07:00~19:00
3        가산안양천점   서울 금천구 가산 디지털2로 127-143, 101호  07:00~20:00
4     가산파트너스타워점    서울특별시 금천구 가산디지털1로 83, 103호-1  07:00~19:30
5    가산하이엔드10차점      서울특별시 금천구 가산디지털1로 30, 102호  07:00~18:30
..          ...                             ...          ...
203        한티역점          서울 강남구 선릉로311, 한티빌딩 1층  07:00~23:00
204        합정역점                   서울 마포구 양화로 72  07:00~23:00
205   홍대입구역사거리점                  서울 마포구 양화로 129  07:00~21:00
206     회기역사거리점         서울 동대문구 회기로 176 (회기동81)  07:00~22:00
207       AK금정점  경기도 군포시 금정동 689번지 AK플라자 금정점 2층  07:30~22:30

[207 rows x 3 columns]
207
